In [50]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import confusion_matrix,classification_report,roc_auc_score

In [51]:
### Load Dataset

In [52]:
df=pd.read_csv("Customer-Churn.csv")

In [53]:
### Basic Cleaning

In [54]:
df=df.drop("customerID",axis=1)
df["TotalCharges"]=pd.to_numeric(df["TotalCharges"],errors="coerce")
df["TotalCharges"]=df["TotalCharges"].fillna(df["TotalCharges"].median())
df["Churn"]=df["Churn"].map({"Yes":1,"No":0})
df.head()


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [55]:
### Split Features and Target

In [56]:
x=df.drop("Churn",axis=1)
y=df["Churn"]

In [57]:
### Identify Column Types

In [67]:
cat_cols=x.select_dtypes(include="object").columns
num_cols=x.select_dtypes(exclude="object").columns

In [68]:
### Preprocessing Pipeline

In [74]:
numeric_pipeline=Pipeline([
    ("imputer",SimpleImputer(strategy="median"))
])

categorical_pipeline=Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))    
])
preprocessor=ColumnTransformer([
    ("num",numeric_pipeline,num_cols),
    ("cat",categorical_pipeline,cat_cols)
])

In [75]:
### Train Test split

In [80]:
x_train,x_test,y_train,y_test=train_test_split(
    x,y,test_size=0.2,stratify=y,random_state=42)
x_train

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
3738,Male,0,No,No,35,No,No phone service,DSL,No,No,Yes,No,Yes,Yes,Month-to-month,No,Electronic check,49.20,1701.65
3151,Male,0,Yes,Yes,15,Yes,No,Fiber optic,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,75.10,1151.55
4860,Male,0,Yes,Yes,13,No,No phone service,DSL,Yes,Yes,No,Yes,No,No,Two year,No,Mailed check,40.55,590.35
3867,Female,0,Yes,No,26,Yes,No,DSL,No,Yes,Yes,No,Yes,Yes,Two year,Yes,Credit card (automatic),73.50,1905.70
3810,Male,0,Yes,Yes,1,Yes,No,DSL,No,No,No,No,No,No,Month-to-month,No,Electronic check,44.55,44.55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6303,Female,0,Yes,No,71,Yes,Yes,Fiber optic,No,Yes,Yes,Yes,Yes,Yes,Two year,No,Electronic check,109.25,7707.70
6227,Male,0,No,No,2,Yes,No,DSL,No,No,No,No,No,No,Month-to-month,No,Bank transfer (automatic),46.05,80.35
4673,Female,1,No,No,25,Yes,Yes,Fiber optic,Yes,Yes,No,No,Yes,Yes,Month-to-month,Yes,Mailed check,102.80,2660.20
2710,Female,0,Yes,No,24,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,No,Credit card (automatic),20.40,482.80


In [81]:
### Apply Preprocessing

In [82]:
x_train_prep=preprocessor.fit_transform(x_train)
x_test_prep=preprocessor.transform(x_test)

In [83]:
### Feature Selection

In [95]:
selector_model=RandomForestClassifier(
    n_estimators=300,
    random_state=42)

selector_model.fit(x_train_prep,y_train)

selector=SelectFromModel(
    selector_model,
    threshold="median",
    prefit=True)

x_train_sel=selector.transform(x_train_prep)
x_test_sel=selector.transform(x_test_prep)

print("Orignal Features: ",x_train_prep.shape[1])
print("selected Features: ",x_train_sel.shape[1])

Orignal Features:  45
selected Features:  23


In [96]:
### Final Random Forst Moldel

In [97]:
model=RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    class_weight="balanced",
    random_state=42
)
model.fit(x_train_sel,y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=500,
                       random_state=42)

In [98]:
### Prediction

In [99]:
pred=model.predict(x_test_sel)
prob=model.predict_proba(x_test_sel)[:,1]

In [100]:
### Evaluation 

In [104]:
print("Confusion Matix:")
print(confusion_matrix(y_test,pred))
print("\nClassification report")
print(classification_report(y_test,pred))
print("\nroc auc:",roc_auc_score(y_test,prob))

Confusion Matix:
[[830 205]
 [117 257]]

Classification report
              precision    recall  f1-score   support

           0       0.88      0.80      0.84      1035
           1       0.56      0.69      0.61       374

    accuracy                           0.77      1409
   macro avg       0.72      0.74      0.73      1409
weighted avg       0.79      0.77      0.78      1409


roc auc: 0.8391588519465758


In [105]:
### Get Feature Names After Encoding

In [106]:
ohe = preprocessor.named_transformers_["cat"]["encoder"]
cat_feature_names = ohe.get_feature_names_out(cat_cols)

# Combine numeric + encoded categorical
all_feature_names = list(num_cols) + list(cat_feature_names)

# Mask of selected features
selected_mask = selector.get_support()

# Selected feature names
selected_features = np.array(all_feature_names)[selected_mask]

print("\nSelected Feature Names:")
for f in selected_features:
    print(f)


Selected Feature Names:
SeniorCitizen
tenure
MonthlyCharges
TotalCharges
gender_Female
gender_Male
Partner_No
Partner_Yes
Dependents_Yes
MultipleLines_No
MultipleLines_Yes
InternetService_Fiber optic
OnlineSecurity_No
OnlineBackup_No
OnlineBackup_Yes
DeviceProtection_No
TechSupport_No
Contract_Month-to-month
Contract_Two year
PaperlessBilling_No
PaperlessBilling_Yes
PaymentMethod_Credit card (automatic)
PaymentMethod_Electronic check
